# Marine Trace AI: train the oil spill detection model
Runtime → Change runtime type → **T4 GPU**, then run the cells from top to bottom (Shift+Enter).

In [ ]:
# 1. Connect Google Drive (click Allow when asked)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Copy the project code and the SOS dataset from Drive and unzip them
!mkdir -p /content/project
!unzip -q -o /content/drive/MyDrive/MarineTraceAI/project_code.zip -d /content/project
!unzip -q -o /content/drive/MyDrive/MarineTraceAI/sos_sentinel.zip -d /content/project
%cd /content/project
!ls Datasets/raw/sos/train/sentinel | head -4
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2b. Restore earlier results from Drive (so new models are compared with the old ones)
!cp -r /content/drive/MyDrive/MarineTraceAI/results/models /content/project/ 2>/dev/null; cp -r /content/drive/MyDrive/MarineTraceAI/results/docs /content/project/ 2>/dev/null; ls /content/project/models


In [ ]:
# 3. Install the segmentation library
!pip install -q segmentation-models-pytorch

In [ ]:
# 4a. Baseline model (simple CNN), about 10-20 minutes
!python -m ml.segmentation.train --dataset sos --arch fcn --epochs 20 --batch 16 --notes "baseline"

In [ ]:
# 4b. Main model: U-Net with ResNet34, about 30-60 minutes
!python -m ml.segmentation.train --dataset sos --arch unet --encoder resnet34 --epochs 30 --batch 16 --notes "main model"

In [ ]:
# 4c. Comparison model: DeepLabV3+ with ResNet50, about 40-70 minutes (optional)
!python -m ml.segmentation.train --dataset sos --arch deeplabv3plus --encoder resnet50 --epochs 25 --batch 16 --notes "comparison"

In [ ]:
# 4d. Transformer model: SegFormer (MiT-B1 encoder), about 30-50 minutes
!pip install -q -U segmentation-models-pytorch
!python -m ml.segmentation.train --dataset sos --arch segformer --encoder mit_b1 --epochs 30 --batch 16 --lr 6e-5 --notes "transformer"


In [ ]:
# 5. Test the best model and save sample prediction images
!python -m ml.segmentation.evaluate --dataset sos --samples 8

In [ ]:
# 6. Save results back to Google Drive (run after EVERY training cell, in case Colab disconnects)
!mkdir -p /content/drive/MyDrive/MarineTraceAI/results
!cp -r models reports docs /content/drive/MyDrive/MarineTraceAI/results/ 2>/dev/null; echo saved
!ls -R /content/drive/MyDrive/MarineTraceAI/results | head -30